In [ ]:
import sys
import cv2
import matplotlib.pyplot as plt
sys.path.append("..")

from pathlib import Path
import pandas as pd
from src.landmarks import extract_landmarks
from src.features import normalize_landmarks

RAW_DIR = Path("../data/raw/UAlpha40 A Comprehensive Dataset of Urdu alphabets for Pakistan Sign Language")

print("Path exists:", RAW_DIR.exists())

Path exists: True


In [2]:
import time

results = []
failed_files = []

start_time = time.time()

# har harf ke folder me se guzarte hain
folders = sorted([f for f in RAW_DIR.iterdir() if f.is_dir() and f.name.endswith("-Original")])

for folder_idx, folder in enumerate(folders):
    label = folder.name.replace("-Original", "")  # jaise "Bay-Original" -> "Bay"

    image_files = [f for f in folder.iterdir() if f.suffix.lower() == ".jpg"]

    for img_path in image_files:
        raw_landmarks = extract_landmarks(img_path)

        if raw_landmarks is None:
            failed_files.append({"file": img_path.name, "label": label})
            continue

        normalized = normalize_landmarks(raw_landmarks)

        # signer ID nikalna filename se (jaise "s0251-10hay.jpg" -> "s0251")
        signer_id = img_path.name.split("-")[0]

        row = {"label": label, "signer_id": signer_id, "filename": img_path.name}
        for i, value in enumerate(normalized):
            row[f"f{i}"] = value

        results.append(row)

    print(f"[{folder_idx + 1}/{len(folders)}] {label}: {len(image_files)} images processed")

elapsed = time.time() - start_time
print(f"\nMukammal! {len(results)} kaamyaab, {len(failed_files)} fail, waqt: {elapsed:.1f} seconds")

[1/36] 1-Hay: 81 images processed
[2/36] Ain: 79 images processed
[3/36] Alif: 81 images processed
[4/36] Bay: 82 images processed
[5/36] Byeh: 80 images processed
[6/36] Chay: 81 images processed
[7/36] Cyeh: 80 images processed
[8/36] Daal: 81 images processed
[9/36] Dal: 78 images processed
[10/36] Dochahay: 80 images processed
[11/36] Fay: 80 images processed
[12/36] Gaaf: 80 images processed
[13/36] Ghain: 80 images processed
[14/36] Hamza: 79 images processed
[15/36] Kaf: 80 images processed
[16/36] Khay: 80 images processed
[17/36] Kiaf: 80 images processed
[18/36] Lam: 80 images processed
[19/36] Meem: 80 images processed
[20/36] Nuun: 80 images processed
[21/36] Nuungh: 80 images processed
[22/36] Pay: 81 images processed
[23/36] Ray: 82 images processed
[24/36] Say: 81 images processed
[25/36] Seen: 82 images processed
[26/36] Sheen: 81 images processed
[27/36] Suad: 82 images processed
[28/36] Taay: 81 images processed
[29/36] Tay: 81 images processed
[30/36] Tuey: 80 images

In [3]:
df_failed = pd.DataFrame(failed_files)
df_failed.to_csv("../data/processed/failed_images.csv", index=False)

print("Total fail:", len(df_failed))
print("\nHar harf me kitni images fail huin:")
print(df_failed["label"].value_counts())

Total fail: 169

Har harf me kitni images fail huin:
label
Tay         38
Ain         13
Ghain       13
Dochahay    11
Sheen        8
Kaf          6
Seen         6
Daal         5
Nuun         5
Pay          5
Taay         5
Alif         4
Tuey         4
Wao          4
Zaal         4
Dal          3
Meem         3
Zuey         3
Byeh         2
Chay         2
Fay          2
Hamza        2
Kiaf         2
Lam          2
Nuungh       2
Ray          2
Say          2
Suad         2
Zaey         2
Zay          2
Zuad         2
1-Hay        1
Cyeh         1
Khay         1
Name: count, dtype: int64


In [4]:
df_landmarks = pd.DataFrame(results)

print("Shape (rows, columns):", df_landmarks.shape)
print("\nColumns ka namoona:", df_landmarks.columns[:6].tolist(), "...")
df_landmarks.head()

Shape (rows, columns): (2727, 66)

Columns ka namoona: ['label', 'signer_id', 'filename', 'f0', 'f1', 'f2'] ...


,label,signer_id,filename,f0,f1,f2,f3,f4,f5,f6,...,f53,f54,f55,f56,f57,f58,f59,f60,f61,f62
0,1-Hay,s0251,s0251-10hay.jpg,0.0,0.0,0.0,0.144057,-0.696173,-0.046720,0.478123,...,-0.447511,1.054861,0.101863,-0.502622,0.969267,-0.009833,-0.403517,0.802021,-0.035178,-0.308149
1,1-Hay,s0252,s0252-10hay.jpg,0.0,0.0,0.0,0.002862,-0.521564,0.121297,0.114996,...,-0.358662,0.855219,-0.409095,-0.287891,0.750940,-0.342864,-0.112112,0.622086,-0.295576,0.031359
2,1-Hay,s0253,s0253-10hay.jpg,0.0,0.0,0.0,0.206797,-0.762612,-0.045245,0.648133,...,-0.435521,1.057891,0.347057,-0.476490,0.969290,0.210020,-0.367453,0.815299,0.191260,-0.263399
3,1-Hay,s0254,s0254-10hay.jpg,0.0,0.0,0.0,0.364359,-0.612799,-0.124997,0.897794,...,-0.612427,1.071940,1.122048,-0.730447,1.032419,0.901794,-0.615665,0.844064,0.755925,-0.490481
4,1-Hay,s0255,s0255-10hay.jpg,0.0,0.0,0.0,0.158055,-0.737145,-0.027868,0.540961,...,-0.428798,0.964727,0.187479,-0.415738,0.902752,0.045193,-0.299164,0.784328,-0.024990,-0.207970


In [6]:
df_landmarks.to_csv("../data/processed/landmarks.csv", index=False)

file_size_mb = Path("../data/processed/landmarks.csv").stat().st_size / 1024 / 1024
print("Saved! File size:", round(file_size_mb, 2), "MB")

Saved! File size: 3.21 MB


In [7]:
# har class me kitni total thi, kitni kaamyaab, kitni fail
total_per_class = pd.Series([f.name.replace("-Original", "") for f in folders]).value_counts()
success_per_class = df_landmarks["label"].value_counts()
fail_per_class = df_failed["label"].value_counts()

summary = pd.DataFrame({
    "success": success_per_class,
    "failed": fail_per_class,
}).fillna(0).astype(int)

summary["fail_rate_%"] = (summary["failed"] / (summary["success"] + summary["failed"]) * 100).round(1)
summary = summary.sort_values("fail_rate_%", ascending=False)

print(summary)
print("\nSabse kam images wali class:", summary["success"].idxmin(), "->", summary["success"].min(), "images")

          success  failed  fail_rate_%
label                                 
Tay            43      38         46.9
Ain            66      13         16.5
Ghain          67      13         16.2
Dochahay       69      11         13.8
Sheen          73       8          9.9
Kaf            74       6          7.5
Seen           76       6          7.3
Nuun           75       5          6.2
Pay            76       5          6.2
Taay           76       5          6.2
Daal           76       5          6.2
Zaal           75       4          5.1
Tuey           76       4          5.0
Wao            76       4          5.0
Alif           77       4          4.9
Meem           77       3          3.8
Dal            75       3          3.8
Zuey           77       3          3.8
Byeh           78       2          2.5
Zuad           78       2          2.5
Nuungh         78       2          2.5
Chay           79       2          2.5
Say            79       2          2.5
Kiaf           78       2

In [8]:
tay_fails = df_failed[df_failed["label"] == "Tay"]["file"].tolist()[:6]

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
axes = axes.flatten()

for ax, fname in zip(axes, tay_fails):
    img_path = RAW_DIR / "Tay-Original" / fname
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(fname, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined